In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv('train.txt', sep=';',header=None,names=['text','emotion'])

In [ ]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [ ]:
df.isnull().sum()

,0
text,0
emotion,0


In [ ]:
unique_emotion=df['emotion'].unique()
emotion_number={}
i=0
for emo in unique_emotion:
    emotion_number[emo]=i
    i+=1

df['emotion']=df['emotion'].map(emotion_number)

In [ ]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [ ]:
df['text']=df['text'].apply(lambda x: x.lower())

In [ ]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [ ]:
import string

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [ ]:
df['text']=df['text'].apply(remove_punctuation)

In [ ]:
def remove_numbers(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new

df['text']=df['text'].apply(remove_numbers)

In [ ]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [ ]:
def remove_emojis(txt):
    new=""
    for i in txt:
      if i.isascii():
        new+=i
    return new

df['text']=df['text'].apply(remove_emojis)

In [ ]:
import nltk

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
stop_words=set(stopwords.words('english'))

In [ ]:
def remove_stopwords(txt):
   words=txt.split()
   cleaned=[]
   for i in words:
    if not i in stop_words:
      cleaned.append(i)

   return ' '.join(cleaned)

In [ ]:
import nltk
nltk.download('punkt_tab')

def remove_stopwords(txt):
   word_tokens=word_tokenize(txt)
   cleaned=[]
   for i in word_tokens:
    if not i in stop_words:
      cleaned.append(i)
   return ' '.join(cleaned)

df['text']=df['text'].apply(remove_stopwords)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(df['text'],df['emotion'],test_size=0.20,random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

### Bag of Words

In [ ]:
bow_vectorizer=CountVectorizer()

x_train_bow=bow_vectorizer.fit_transform(x_train)
x_test_bow=bow_vectorizer.transform(x_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [ ]:
nb_model=MultinomialNB()
nb_model.fit(x_train_bow,y_train)

MultinomialNB()

In [ ]:
pred_bow = nb_model.predict(x_test_bow)

In [ ]:
print(accuracy_score(y_test,pred_bow))
print(confusion_matrix(y_test,pred_bow))

0.7678125
[[899   8   3   0   4  32]
 [ 87 271   0   0  13  56]
 [ 51   6  78   0   4 157]
 [ 40   0   1   6  17  49]
 [ 87  15   0   0 226  69]
 [ 36   4   2   0   2 977]]


### TF-IDF

In [ ]:
tfidf_vectorizer=TfidfVectorizer()

x_train_tfidf=tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf=tfidf_vectorizer.transform(x_test)

In [ ]:
nb_model_2=MultinomialNB()
nb_model_2.fit(x_train_bow,y_train)

MultinomialNB()

In [ ]:
pred_tfidf= nb_model_2.predict(x_test_tfidf)

In [ ]:
print(accuracy_score(y_test,pred_tfidf))
print(confusion_matrix(y_test,pred_tfidf))

0.7275
[[892   4   1   0   3  46]
 [111 226   0   0   9  81]
 [ 47   3  50   0   2 194]
 [ 44   0   0   3   6  60]
 [116  15   0   0 170  96]
 [ 32   2   0   0   0 987]]


In [ ]:
from sklearn.linear_model import LogisticRegression

Log_model=LogisticRegression(max_iter=1000)
Log_model.fit(x_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [ ]:
pred_tfidf_log= Log_model.predict(x_test_tfidf)

In [ ]:
print(accuracy_score(y_test,pred_tfidf_log))
print(confusion_matrix(y_test,pred_tfidf_log))

0.8615625
[[892  13   1   0   7  33]
 [ 30 348   1   0  12  36]
 [ 12   5 180   0   4  95]
 [ 15   0   1  53  24  20]
 [ 24  18   1   7 300  47]
 [ 15   2  17   0   3 984]]
